<a href="https://colab.research.google.com/github/ilincabaiasu/IB9AU/blob/main/Task_17.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Required Task 17

*Note: GitHub does not support live outputs, please open Colab for running the code.*

### Details

You are a Quantitative Analyst. Your boss wants to know which sector
has performed better this year on a risk-adjusted basis: Big Tech or Big Banks.

Task: Write a prompt for your Local smolagents (Qwen 3B) agent to perform the
following steps autonomously:
Data Ingestion: Download daily closing prices for the last 180 days for a Tech Portfolio
(NVDA, AAPL, MSFT) and a Bank Portfolio (JPM, BAC, C).

Financial Math:
Calculate the Daily Returns for each stock.
Calculate the Sharpe Ratio for each stock (Assume risk-free rate = 0, so simply Mean
Daily Return / Std Dev of Daily Returns * sqrt(252)).


Visualization:
Create a Bar Chart comparing the Sharpe Ratios of all 6 companies.
Color code the bars: Green for Tech, Blue for Banks.

Output: Save the chart as sharpe_comparison.png.

*Note: The Qwen-3B model was unable to reliably complete a 6-step multi-output task autonomously, producing hallucinated outputs, syntax errors, and narration instead of code execution. The analysis was completed using the fallback script. This illustrates a key limitation of small local models for complex agentic tasks; larger models (7B+) or task decomposition into single-step agent calls would be needed for reliable autonomous execution.*

### What I found interesting:
- The prompt is the programme
- The agent writes and executes its own code
- Able to observe the thought-code-observation loop live

In [ ]:
!pip install -q smolagents transformers accelerate bitsandbytes yfinance matplotlib numpy pandas
print(" Dependencies installed.")

In [ ]:
from smolagents import CodeAgent, TransformersModel
import torch

print(" Loading Qwen2.5-Coder-3B-Instruct (~6GB)...")

model = TransformersModel(
    model_id="Qwen/Qwen2.5-Coder-3B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16,
    max_new_tokens=2048
)

print(" Model loaded on GPU.")

### Initialise Code Agent

In [ ]:
agent = CodeAgent(
    tools=[],
    model=model,
    max_steps=5,   # allow extra steps for this multi-stage task
    additional_authorized_imports=[
        "yfinance",
        "pandas",
        "numpy",
        "matplotlib",
        "matplotlib.pyplot",
    ]
)

print(" CodeAgent initialised.")

### Sharpe Ratio Comparison Prompt

In [ ]:
task_prompt = """
You are a quantitative analyst. Complete the following steps exactly:

STEP 1 — DATA INGESTION:
Use yfinance to download daily closing prices for the last 180 days for these 6 tickers:
  Tech Portfolio : NVDA, AAPL, MSFT
  Bank Portfolio : JPM, BAC, C
Use yf.download() with auto_adjust=True and keep only the 'Close' column.

STEP 2 — DAILY RETURNS:
The 'Close' DataFrame has one column per ticker.
Calculate daily returns by calling .pct_change() on the entire DataFrame at once:
    returns = raw.pct_change().dropna()
Do NOT create a separate DataFrame per stock.
Do NOT use a 'Daily_Return' column.
Access each stock's returns as returns['NVDA'], returns['AAPL'] etc.

STEP 3 — SHARPE RATIO:
For each of the 6 stocks, calculate the annualised Sharpe Ratio using this formula:
  Sharpe = (Mean Daily Return / Std Dev of Daily Returns) * sqrt(252)
Assume risk-free rate = 0.
Store the results in a dictionary: {ticker: sharpe_ratio}.
Print the Sharpe Ratio for each stock.

STEP 4 — BAR CHART:
Create a bar chart of the Sharpe Ratios for all 6 companies.
Color-code the bars:
  - Green (#2ecc71) for Tech stocks: NVDA, AAPL, MSFT
  - Blue  (#2980b9) for Bank stocks: JPM, BAC, C
Add a horizontal dashed red line at y=0.
Add bar value labels on top of each bar rounded to 2 decimal places.
Title: 'Sharpe Ratio: Big Tech vs Big Banks (Last 180 Days)'
X-axis label: 'Stock'
Y-axis label: 'Annualised Sharpe Ratio (rf=0)'
Add a legend showing 'Tech (Green)' and 'Banks (Blue)'.
Use tight_layout().

STEP 5 — SAVE:
Save the chart as 'sharpe_comparison.png' with dpi=150.
Print: 'Chart saved as sharpe_comparison.png'

STEP 6 — CONCLUSION:
Based on the Sharpe Ratios, print a one-sentence conclusion stating which sector
(Tech or Banks) has delivered better risk-adjusted returns over the last 180 days
and why (higher Sharpe = better risk-adjusted return).
"""

print("🤖 Agent is coding... (watch the Thought/Code/Observation loop below)\n")
result = agent.run(task_prompt, stream=False)

The error is not fixable from our side, but a limitation of the model's context window and reasoning

Agent's Conclusion

In [ ]:
print("\n AGENT CONCLUSION:")
print(result)

In [ ]:
import IPython
import os

if os.path.exists("sharpe_comparison.png"):
    print("📊 Sharpe Ratio Comparison Chart:")
    IPython.display.display(IPython.display.Image("sharpe_comparison.png"))
else:
    print("⚠️ sharpe_comparison.png not found.")
    print("The agent may have encountered an error — check the output logs above.")
    print("If needed, run the fallback cell below.")

Run the analysis directly

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os, IPython

tech_tickers = ['NVDA', 'AAPL', 'MSFT']
bank_tickers = ['JPM', 'BAC', 'C']
all_tickers  = tech_tickers + bank_tickers

raw = yf.download(all_tickers, period='180d', auto_adjust=True)
if isinstance(raw.columns, pd.MultiIndex):
    raw = raw['Close']

returns = raw.pct_change().dropna()

sharpe = {}
for ticker in all_tickers:
    col = returns[ticker].squeeze()
    sharpe[ticker] = (float(col.mean()) / float(col.std())) * np.sqrt(252)

for ticker, sr in sharpe.items():
    print(f"{ticker}: {sr:.4f}")

colors = ['#2ecc71' if t in tech_tickers else '#2980b9' for t in all_tickers]
values = [float(sharpe[t]) for t in all_tickers]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(all_tickers, values, color=colors, edgecolor='white')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.02, f'{val:.2f}',
            ha='center', fontsize=11, fontweight='bold')

ax.axhline(0, color='red', linestyle='--', linewidth=1.2)
ax.legend(handles=[
    mpatches.Patch(color='#2ecc71', label='Tech (NVDA, AAPL, MSFT)'),
    mpatches.Patch(color='#2980b9', label='Banks (JPM, BAC, C)')
])
ax.set_title('Sharpe Ratio: Big Tech vs Big Banks (Last 180 Days)', fontweight='bold')
ax.set_xlabel('Stock')
ax.set_ylabel('Annualised Sharpe Ratio (rf=0)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('sharpe_comparison.png', dpi=150)
plt.show()
print("Chart saved as sharpe_comparison.png")

avg_tech = np.mean([sharpe[t] for t in tech_tickers])
avg_bank = np.mean([sharpe[t] for t in bank_tickers])
winner = 'Tech' if avg_tech > avg_bank else 'Banks'
print(f"\nAverage Sharpe — Tech : {avg_tech:.4f}")
print(f"Average Sharpe — Banks: {avg_bank:.4f}")
print(f"\nCONCLUSION: {winner} has delivered better risk-adjusted returns over the last 180 days.")

if os.path.exists('sharpe_comparison.png'):
    IPython.display.display(IPython.display.Image('sharpe_comparison.png'))